# T1.4 Colab 확인: 패치 풀·테스트셋 LR 생성 (실제 DIV2K 800장)

`scripts/prepare_data.py`를 실제 데이터로 실행하고 산출물을 확인합니다. 로직은 `src/sr_pipeline/data/prepare.py`에 있고 이 노트북은 실행·확인만 합니다.

- 셀 1~2: 저장소(YongMin 브랜치) 가져오기, 최신 코드 확인
- 셀 3: **Linux에서 `patch_pool_id` 재현 확인** (로컬 Windows에서 만든 기준 값과 비교)
- 셀 4: DIV2K(ETH 공식)와 테스트셋 HR(Hugging Face 공개 미러)을 받고 영상 수 확인
- 셀 5: 실제 데이터로 압축 파일 생성
- 셀 6: 압축 해제 후 산출물 검증
- 셀 7: 기록할 요약 (이 출력을 그대로 붙여 달라고 전달)

GPU는 필요 없습니다(CPU 런타임이어도 됩니다). 토큰은 필요 없습니다(저장소는 Public).


In [ ]:
import os
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull --ff-only          # 이미 받아 둔 경우 최신으로
else:
    !git clone --branch YongMin --depth 1 {REPO_URL} /content/repo
!git -C /content/repo log --oneline -4             # T1.4 커밋 2개(feat/fix ... (T1.4))가 보여야 함. 안 보이면 로컬에서 git push 먼저
assert os.path.isdir("/content/repo/src/sr_pipeline"), "clone 실패: REPO_URL/푸시 확인"
print("셀 1 OK")

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/repo/src")
sys.path_importer_cache.pop("/content/repo/src", None)
importlib.invalidate_caches()
# 이전 실행에서 import한 옛 모듈이 커널 메모리에 남아 있으면 git pull 후에도 옛 코드가 실행된다 -> 캐시를 비운다.
for name in [m for m in sys.modules if m == "sr_pipeline" or m.startswith("sr_pipeline.")]:
    del sys.modules[name]
from sr_pipeline.data import prepare
assert hasattr(prepare, "prepare_dataset_archive"), \
    "prepare.py가 옛 버전입니다: 로컬에서 git push 한 뒤 셀 1을 다시 실행하고 런타임을 재시작하세요."
import platform, torch, numpy as np, PIL
print("platform", platform.platform())
print("torch", torch.__version__, "| numpy", np.__version__, "| PIL", PIL.__version__)
print("셀 2 OK")

In [ ]:
# Linux(Colab)에서도 같은 입력이 같은 patch_pool_id를 만드는지 본다. 합성 영상은 시드 고정 잡음 PNG라 어디서나 같다.
# 기준 값은 로컬 Windows(torch 2.15.0+cpu, numpy 2.4.6, PIL 13.0.0)에서 같은 코드로 만든 값이다.
# 다르면 FAIL이 아니라 "OS·라이브러리 버전에 따라 풀이 달라질 수 있다"는 증거이므로, 값과 버전을 그대로 기록한다.
import tempfile
from pathlib import Path
from PIL import Image

EXPECTED_SYNTHETIC_ID = "div2k_x4_p48_n20_s42_58d3264d"
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    rng = np.random.default_rng(0)
    (root / "hr").mkdir()
    for i in range(5):
        Image.fromarray(rng.integers(0, 256, (120, 150, 3), dtype=np.uint8)).save(root / "hr" / f"{i:04d}.png")
    synthetic = prepare.create_patch_pool(
        root / "hr", root / "out", scale=4, patch_size_hr=48, num_patches=20, pool_seed=42,
        degradation="bicubic", bicubic_rule="matlab_imresize_antialias")
print("이 런타임:", synthetic.patch_pool_id)
print("기준(Windows):", EXPECTED_SYNTHETIC_ID)
print("OS 간 재현:", "OK 동일" if synthetic.patch_pool_id == EXPECTED_SYNTHETIC_ID else "DIFFERENT (기록할 것)")

In [ ]:
# DIV2K 학습셋 HR(800장)을 받는다. 이미 있으면 건너뛴다. (약 3.5GB, Colab에서 수 분)
DIV2K_HR_DIR = "/content/raw/DIV2K_train_HR"
if not os.path.isdir(DIV2K_HR_DIR):
    !mkdir -p /content/raw
    !wget -q -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -P /content/raw
    !unzip -q -n /content/raw/DIV2K_train_HR.zip -d /content/raw

# --- 테스트셋 HR 받기 시작 ---
# 공개 미러(Hugging Face eugenesiow/<이름>)의 <이름>_HR.tar.gz를 받아 <데이터셋>/<영상>.png 구조로 만든다.
# (압축 안에는 <이름>_HR/ 폴더 하나, 영상은 PNG: Set5 5장, Set14 14장, BSD100 100장, Urban100 100장. 합계 약 170MB)
# 이미 해당 폴더가 있으면 건너뛴다. 다른 출처를 쓰려면 TESTSETS_HR_DIR/<이름>/*.png 구조로 직접 준비하면 된다.
import tarfile, tempfile, shutil, urllib.request
from pathlib import Path

TESTSETS_HR_DIR = "/content/raw/testsets"
for name in ("Set5", "Set14", "BSD100", "Urban100"):
    target = Path(TESTSETS_HR_DIR) / name
    if target.is_dir():
        print(name, ": 이미 있음, 건너뜀")
        continue
    url = f"https://huggingface.co/datasets/eugenesiow/{name}/resolve/main/data/{name}_HR.tar.gz"
    with tempfile.TemporaryDirectory() as tmp:
        archive = Path(tmp) / f"{name}_HR.tar.gz"
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response, open(archive, "wb") as out:
            shutil.copyfileobj(response, out)
        with tarfile.open(archive) as tar:
            try:
                tar.extractall(tmp, filter="data")      # 경로 탈출·특수 파일을 막는 안전 추출(Python 3.11.4+)
            except TypeError:
                tar.extractall(tmp)                     # 옛 Python: 압축 내용은 위에서 확인한 일반 PNG뿐
        extracted = Path(tmp) / f"{name}_HR"
        assert extracted.is_dir(), f"{name}: 압축 안에 {name}_HR 폴더가 없습니다(미러 구조가 바뀜). 직접 준비하세요."
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(extracted), str(target))
    print(name, ": 받음")
# --- 테스트셋 HR 받기 끝 ---

EXPECTED_DIV2K_COUNT = 800
EXPECTED_TESTSET_COUNTS = {"Set5": 5, "Set14": 14, "BSD100": 100, "Urban100": 100}

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
def count_images(directory):
    return sum(1 for p in Path(directory).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

n_div2k = count_images(DIV2K_HR_DIR)
print("DIV2K train HR:", n_div2k)
assert n_div2k == EXPECTED_DIV2K_COUNT, f"DIV2K 영상 수가 {EXPECTED_DIV2K_COUNT}장이 아닙니다: {n_div2k}"
for name, expected in EXPECTED_TESTSET_COUNTS.items():
    folder = Path(TESTSETS_HR_DIR) / name
    assert folder.is_dir(), f"{folder} 없음: 테스트셋을 <데이터셋>/<영상> 구조로 TESTSETS_HR_DIR에 준비하세요."
    n = count_images(folder)
    print(f"{name}: {n}")
    assert n == expected, f"{name} 영상 수가 {expected}장이 아닙니다: {n} (하위 폴더에 영상이 있으면 바로 아래로 옮겨야 함)"
print("셀 4 OK")

In [ ]:
import subprocess, sys, time

# 파일럿용 값이다(공정 비교 조건이 아님). 본 실행 풀은 T4.2에서 확정한 값으로 다시 만든다.
PATCH_SIZE_HR = 96
NUM_PATCHES = 10000
POOL_SEED = 42
OUT_DIR = "/content/data"
ARCHIVE_PATH = "/content/archive/div2k_testsets_x4.zip"

for path in (OUT_DIR, ARCHIVE_PATH):
    assert not os.path.exists(path), f"{path}가 이미 있습니다. 덮어쓰지 않으니 지우고(!rm -rf {path}) 다시 실행하세요."

cmd = [
    sys.executable, "scripts/prepare_data.py",   # 커널과 같은 인터프리터(torch가 설치된 Colab Python)
    "--div2k-hr-dir", DIV2K_HR_DIR, "--testsets-hr-dir", TESTSETS_HR_DIR,
    "--output-dir", OUT_DIR, "--archive-path", ARCHIVE_PATH,
    "--patch-size-hr", str(PATCH_SIZE_HR), "--num-patches", str(NUM_PATCHES), "--pool-seed", str(POOL_SEED),
]
start = time.time()
result = subprocess.run(cmd, cwd="/content/repo", env={**os.environ, "PYTHONPATH": "src"},
                        capture_output=True, text=True)
elapsed = time.time() - start
print(result.stdout)
print(result.stderr[-3000:])
assert result.returncode == 0, "prepare_data.py 실패 (위 stderr 확인)"
print(f"셀 5 OK: {elapsed:.0f}초")

In [ ]:
import json, zipfile
from pathlib import Path
from PIL import Image
from sr_pipeline.data.prepare import _content_sha256, degrade

UNPACK_DIR = "/content/unpacked"
assert not os.path.exists(UNPACK_DIR), f"{UNPACK_DIR}가 이미 있습니다: 지우고(!rm -rf {UNPACK_DIR}) 다시 실행하세요."
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None, "압축 파일이 손상되었습니다"
    archive.extractall(UNPACK_DIR)

pool_dir = Path(UNPACK_DIR) / "patch_pool"
meta = json.loads((pool_dir / "metadata.json").read_text(encoding="utf-8"))
hr = np.load(pool_dir / "hr.npy")
lr = np.load(pool_dir / "lr.npy")

results = {}
results["hr/lr 배열 dtype·shape"] = (
    hr.dtype == np.uint8 and lr.dtype == np.uint8
    and hr.shape == (NUM_PATCHES, PATCH_SIZE_HR, PATCH_SIZE_HR, 3)
    and lr.shape == (NUM_PATCHES, PATCH_SIZE_HR // 4, PATCH_SIZE_HR // 4, 3))
results["압축 해제 후 content_sha256 재계산 일치"] = _content_sha256(hr, lr) == meta["content_sha256"]
results["patch_pool_id 접미사 = content_sha256[:8]"] = (
    meta["patch_pool_id"] == f"div2k_x4_p{PATCH_SIZE_HR}_n{NUM_PATCHES}_s{POOL_SEED}_{meta['content_sha256'][:8]}")
results[f"원천 영상 수 {EXPECTED_DIV2K_COUNT}"] = meta["source_image_count"] == EXPECTED_DIV2K_COUNT
results["생성 조건 기록 일치"] = (
    (meta["scale"], meta["degradation"], meta["bicubic_rule"],
     meta["patch_size_hr"], meta["num_patches"], meta["pool_seed"])
    == (4, "bicubic", "matlab_imresize_antialias", PATCH_SIZE_HR, NUM_PATCHES, POOL_SEED))

# 테스트셋: 영상 수, HR이 4의 배수, LR = HR/4, 저장된 LR = 같은 열화를 HR에 다시 적용한 결과
testsets_meta = json.loads((Path(UNPACK_DIR) / "testsets" / "metadata.json").read_text(encoding="utf-8"))
results["테스트셋 영상 수 기록"] = testsets_meta["image_counts"] == EXPECTED_TESTSET_COUNTS
for name, expected in EXPECTED_TESTSET_COUNTS.items():
    hr_files = sorted(p.name for p in (Path(UNPACK_DIR) / "testsets" / name / "HR").iterdir())
    lr_files = sorted(p.name for p in (Path(UNPACK_DIR) / "testsets" / name / "LR").iterdir())
    ok = len(hr_files) == expected and hr_files == lr_files
    mismatched = []
    for file_name in hr_files:
        hr_image = np.array(Image.open(Path(UNPACK_DIR) / "testsets" / name / "HR" / file_name).convert("RGB"))
        lr_image = np.array(Image.open(Path(UNPACK_DIR) / "testsets" / name / "LR" / file_name).convert("RGB"))
        good = (hr_image.shape[0] % 4 == 0 and hr_image.shape[1] % 4 == 0
                and lr_image.shape == (hr_image.shape[0] // 4, hr_image.shape[1] // 4, 3)
                and np.array_equal(lr_image, degrade(hr_image, 4)))
        if not good:
            mismatched.append(file_name)
    results[f"{name}: {expected}장, HR/LR 쌍·크기·열화 일치"] = ok and not mismatched
    if mismatched:
        print("불일치 영상:", name, mismatched[:5])

for name, ok in results.items():
    print("OK  " if ok else "FAIL", name)
print("전체:", "PASS" if all(results.values()) else "FAIL")
assert all(results.values())

In [ ]:
# 이 출력을 그대로 복사해 전달한다(T1.4 통과 근거, T1.5 입력).
size_mb = os.path.getsize(ARCHIVE_PATH) / 1e6
print("patch_pool_id          :", meta["patch_pool_id"])
print("content_sha256         :", meta["content_sha256"])
print("source_image_count     :", meta["source_image_count"])
print("patch_size_hr / num    :", meta["patch_size_hr"], "/", meta["num_patches"], "(pool_seed", meta["pool_seed"], ")")
print("archive                :", ARCHIVE_PATH, f"({size_mb:.0f} MB)")
print("elapsed (prepare_data) :", f"{elapsed:.0f}초")
print("synthetic id vs Windows:", "동일" if synthetic.patch_pool_id == EXPECTED_SYNTHETIC_ID else "다름: " + synthetic.patch_pool_id)
print("platform               :", platform.platform())
print("versions               : torch", torch.__version__, "| numpy", np.__version__, "| PIL", PIL.__version__)
!git -C /content/repo log --oneline -1

In [ ]:
# 압축 파일은 런타임(/content)에만 있어 세션이 끝나면 사라진다. Drive로 복사하고 해시로 같은 파일인지 확인한다. (T1.5의 첫 단계)
import hashlib, shutil
from google.colab import drive
drive.mount("/content/drive")

# 팀이 공유하는 폴더로 바꿔도 된다(공유 폴더의 바로가기를 내 드라이브에 추가하면 MyDrive 아래에서 보인다).
DRIVE_DIR = "/content/drive/MyDrive/sr_project_data"
DRIVE_PATH = os.path.join(DRIVE_DIR, f"{meta['patch_pool_id']}.zip")   # 파일명에 patch_pool_id를 넣어 풀이 섞이지 않게 한다
os.makedirs(DRIVE_DIR, exist_ok=True)
assert not os.path.exists(DRIVE_PATH), f"{DRIVE_PATH}가 이미 있습니다. 덮어쓰지 않으니 확인 후 지우고 다시 실행하세요."

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

shutil.copy2(ARCHIVE_PATH, DRIVE_PATH)
assert sha256_of(ARCHIVE_PATH) == sha256_of(DRIVE_PATH), "Drive 사본이 원본과 다릅니다: 다시 복사하세요."
print("Drive 사본 OK:", DRIVE_PATH, f"({os.path.getsize(DRIVE_PATH) / 1e6:.0f} MB)")

# 내 PC로 직접 받으려면(브라우저 Colab에서는 동작, VS Code 확장에서는 안 될 수 있음):
# from google.colab import files; files.download(ARCHIVE_PATH)